离线构造教学语言视图。核心代码、已有输出与配置沿已接受读者源；固定申报事实/图属于历史教学输入。下列新计算不构成市场策略复现。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch08_regression_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第8章：回归、稳健推断与正则化

全部为构造数据，不是CAPM实证。生成过程的日截距.0001、斜率1.2，误差尺度随市场变化且有AR(1)依赖。400个等间隔UTC日时间戳中前280行训练，末120行只作最后一次预测误差评估。

OLS含截距，比较经典、HC3、HAC标准误；再造近重复特征比较Ridge/Lasso/ElasticNet。超参数只在训练窗内以gap=5的TimeSeriesSplit选择，StandardScaler置于Pipeline内，留出不参与选择。接口校验时区、唯一递增时间、目标/特征精确对齐、有限值、非常数与满秩。

## 继承的构造运行（seed20261008）

训练alpha=.00064027/日、beta=1.124669、R²=.776925、调整R²=.776123；斜率标准误经典.036144、HC3 .052391、HAC5 .055402。近重复设计条件数135724。选择Ridge alpha=.01、Lasso alpha=.00001、ElasticNet alpha=.00001且l1_ratio=.2。120行留出MSE：均值.0002291795、OLS .0000400386、Ridge .0000400437、Lasso .0000400481、ElasticNet .0000400414（日收益平方）。这些是继承运行记录，本次未重算，不证明市场预测能力。

## 1. 设置与固定种子样本

In [ ]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.dates import AutoDateLocator, ConciseDateFormatter
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
from IPython.display import Image, display
from quant_course.regression_tools import (
    fit_ols_with_robust_covariance,
    select_regularized_models,
    simulate_capm_like_data,
)

SEED = 20261008
TRAIN_N = 280
HAC_MAXLAGS = 5
CV_GAP = 5
Y_UNIT = "Excess return (decimal per day)"
data = simulate_capm_like_data(n=400, seed=SEED, alpha=0.0001, beta=1.2, ar_phi=0.55)
training = data.iloc[:TRAIN_N]
holdout = data.iloc[TRAIN_N:]
print("CONSTRUCTED data only; no CAPM empirical test")
print(f"seed={SEED}; n={len(data)}; frequency=one synthetic daily observation")
print(f"train n={len(training)}: {training.index[0]} to {training.index[-1]}")
print(f"holdout n={len(holdout)}: {holdout.index[0]} to {holdout.index[-1]}")
print("true DGP alpha=0.0001 return/day; beta=1.2; residual has conditional heteroskedasticity and AR(1) phi=0.55")


## 2. OLS系数与协方差估计

三种协方差选择的点估计相同；经典标准误依赖经典误差条件，HC3处理异方差，HAC在额外条件下处理异方差和序列依赖。本例固定maxlags=5。

In [ ]:
training_features = training[["market_excess"]]
training_target = training["asset_excess"]
ols = fit_ols_with_robust_covariance(training_features, training_target, hac_maxlags=HAC_MAXLAGS)
classical = ols.classical

inference_rows = []
for label, result in (("Classical", ols.classical), ("HC3", ols.hc3), ("HAC, maxlags=5", ols.hac)):
    inference_rows.append({
        "covariance": label,
        "alpha estimate": float(np.asarray(result.params)[0]),
        "beta estimate": float(np.asarray(result.params)[1]),
        "beta SE": float(np.asarray(result.bse)[1]),
        "beta t": float(np.asarray(result.tvalues)[1]),
        "beta p": float(np.asarray(result.pvalues)[1]),
        "joint slope F": float(result.fvalue),
        "F p": float(result.f_pvalue),
    })
inference_table = pd.DataFrame(inference_rows).set_index("covariance")
inference_display = inference_table.copy()
for p_value_column in ("beta p", "F p"):
    inference_display[p_value_column] = inference_display[p_value_column].map(lambda value: f"{value:.3e}")
display(inference_display.round(6))
print(f"training n={int(classical.nobs)}; R2={classical.rsquared:.6f}; adjusted R2={classical.rsquared_adj:.6f}; condition={ols.design_condition_number:.3f}")
print("t/F null for the slope: beta=0. Classical inference assumes homoskedastic, serially independent errors; the constructed errors violate these assumptions.")
print("HAC maxlags=5 is a fixed teaching choice, not a bandwidth optimum.")


## 3. 共线性怎样改变系数分配

第二特征=构造市场收益加微小噪声，并非另一真实市场因子。条件数度量数值敏感性；个别斜率可大幅变化，拟合值却类似。系数是指定模型的条件相关，不识别因果。

In [ ]:
near_duplicate_fit = fit_ols_with_robust_covariance(
    training[["market_excess", "market_near_duplicate"]],
    training["asset_excess"],
    hac_maxlags=HAC_MAXLAGS,
)
print(f"near-duplicate design condition number={near_duplicate_fit.design_condition_number:.3f}")
print("two-feature OLS slopes:", near_duplicate_fit.classical.params.iloc[1:].to_numpy())
print("one-feature OLS slope:", classical.params["market_excess"])
print("Specification-dependent coefficient differences are not evidence of causal effects.")


## 4. 仅用过去训练资料选择正则化

三个搜索仅见前280行；每折留5行gap，缩放只在该折训练片拟合。选定后在完整280行重拟合，不用留出选参。

In [ ]:
train_features = training[["market_excess", "market_near_duplicate"]]
selected = select_regularized_models(
    train_features,
    training_target,
    n_splits=5,
    gap=CV_GAP,
    test_size=30,
)
cv_rows = []
for name, result in selected.items():
    cv_rows.append({"model": name, "best params": result.best_params, "mean CV MSE": result.best_cv_mse})
    assert result.training_n == TRAIN_N
    assert result.training_end == training.index[-1]
    assert all(fold.gap_n == CV_GAP for fold in result.folds)
display(pd.DataFrame(cv_rows).set_index("model").round(10))
print("fold boundaries (same across models):")
for number, fold in enumerate(next(iter(selected.values())).folds, start=1):
    print(f"fold {number}: train n={fold.train_n} through {fold.train_end.date()}, gap={fold.gap_n}, validate n={fold.validation_n} from {fold.validation_start.date()}")


## 5. 最后一次留出评估

现在才使用保留的120行计算MSE。此单次比较既不把模拟变为实证，也不据此选冠军。

In [ ]:
holdout_features = holdout[["market_excess", "market_near_duplicate"]]
ols_holdout_features = sm.add_constant(holdout[["market_excess"]], has_constant="add")
predictions = {
    "Mean baseline": np.full(len(holdout), float(training_target.mean())),
    "OLS (one factor)": ols.classical.predict(ols_holdout_features).to_numpy(),
}
for name, result in selected.items():
    predictions[name] = result.estimator.predict(holdout_features)
holdout_mse = {
    name: float(mean_squared_error(holdout["asset_excess"], predicted))
    for name, predicted in predictions.items()
}
mse_table = pd.DataFrame({"n": len(holdout), "holdout MSE (return/day)^2": holdout_mse})
display(mse_table)
print("No hyperparameter or model was chosen from the holdout results.")
print("MSE is forecast error on constructed observations, not net strategy performance.")


## 6. 拟合值、残差与留出预测

新画四面板：构造过程与训练拟合、残差诊断、留出预测以及MSE（小数日收益平方），保存到本次RUN_DIR。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), constrained_layout=True)
fitted = ols.classical.fittedvalues
residuals = ols.classical.resid
axes[0, 0].plot(training.index, training["asset_excess"], color="#235789", linewidth=1.0, label="Observed")
axes[0, 0].plot(training.index, fitted, color="#C1292E", linewidth=1.0, label="OLS fitted")
axes[0, 0].set_title(f"Training observations and OLS fit (n={len(training)})")
axes[0, 0].set_ylabel(Y_UNIT)
axes[0, 0].set_xlabel("UTC observation date")
axes[0, 0].legend(frameon=False)
axes[0, 0].grid(alpha=0.2)

axes[0, 1].scatter(fitted, residuals, color="#235789", s=18, alpha=0.65)
axes[0, 1].axhline(0.0, color="#C1292E", linestyle="--", linewidth=1.0)
axes[0, 1].set_title("Training residual diagnostic")
axes[0, 1].set_xlabel("OLS fitted excess return (decimal per day)")
axes[0, 1].set_ylabel("OLS residual (decimal per day)")
axes[0, 1].grid(alpha=0.2)

colors = {"Mean baseline":"#6C757D", "OLS (one factor)":"#C1292E", "Ridge":"#2A9D8F", "Lasso":"#E9C46A", "ElasticNet":"#8A5FBF"}
axes[1, 0].plot(holdout.index, holdout["asset_excess"], color="#1D3557", linewidth=1.3, label="Observed")
for name, values in predictions.items():
    axes[1, 0].plot(holdout.index, values, color=colors[name], linewidth=0.95, label=name)
axes[1, 0].set_title(f"Final chronological holdout predictions (n={len(holdout)})")
axes[1, 0].set_ylabel(Y_UNIT)
axes[1, 0].set_xlabel("UTC observation date")
axes[1, 0].legend(frameon=False, ncol=2, fontsize=8)
axes[1, 0].grid(alpha=0.2)

labels = list(holdout_mse)
compact_labels = {"Mean baseline":"Mean", "OLS (one factor)":"OLS-1F", "Ridge":"Ridge", "Lasso":"Lasso", "ElasticNet":"ElasticNet"}
axes[1, 1].bar([compact_labels[label] for label in labels], [holdout_mse[label] for label in labels], color=[colors[label] for label in labels])
axes[1, 1].set_title("Holdout MSE; lower is smaller")
axes[1, 1].set_ylabel("MSE ((decimal return per day)²)")
axes[1, 1].grid(axis="y", alpha=0.2)
fig.suptitle(f"Chapter 8 — constructed observations only; seed {SEED}; train n={len(training)}, holdout n={len(holdout)}")
for time_axis in (axes[0, 0], axes[1, 0]):
    locator = AutoDateLocator(minticks=4, maxticks=6)
    time_axis.xaxis.set_major_locator(locator)
    time_axis.xaxis.set_major_formatter(ConciseDateFormatter(locator))
figure_path = RUN_DIR / "figures" / "a2" / "ch08_regression.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(figure_path)))
print(f"Saved figure: {figure_path.relative_to(ROOT)}")


## 检查与限制

无噪声小例应在浮点容差内恢复截距和斜率。经典t/F推断需模型条件；HC3/HAC适用范围及滞后选择仍需判断。正则化可稳定共线系数，但标准化系数不同于原特征单位。TimeSeriesSplit使用等间隔时点，未使用打乱K折。系数符号/显著性不识别因果；这里无真实证券、真实市场/无风险序列、组合或费用模型，不证明可交易性。